# Gemini Image Retrieval Results Review

Offline review notebook for inspecting image-query retrieval results from `data/eval/gemini_retrieval_report.json`.

This notebook does not call the Gemini API.

In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import numpy as np
from IPython.display import HTML, Image, display

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

QUERY_PATH = ROOT / "data/eval/image_queries.jsonl"
PRODUCT_PATH = ROOT / "data/extracted/product_parse_sample.jsonl"
REPORT_PATH = ROOT / "data/eval/gemini_retrieval_report.json"
PRODUCT_EMBEDDING_DIR = ROOT / "data/embeddings/gemini"
QUERY_EMBEDDING_DIR = ROOT / "data/embeddings/gemini/queries/image"

ROOT

In [ ]:
def load_jsonl(path: Path) -> list[dict]:
    return [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line]


queries = {record["query_id"]: record for record in load_jsonl(QUERY_PATH)}
products = {record["product_id"]: record for record in load_jsonl(PRODUCT_PATH)}
report = json.loads(REPORT_PATH.read_text(encoding="utf-8"))["image"]
product_embedding_metadata = load_jsonl(PRODUCT_EMBEDDING_DIR / "metadata.jsonl")
query_embedding_metadata = load_jsonl(QUERY_EMBEDDING_DIR / "metadata.jsonl")
product_ids_by_embedding_row = [record["product_id"] for record in product_embedding_metadata]
query_ids_by_embedding_row = [record["query_id"] for record in query_embedding_metadata]
product_embedding_rows = {product_id: index for index, product_id in enumerate(product_ids_by_embedding_row)}
query_embedding_rows = {query_id: index for index, query_id in enumerate(query_ids_by_embedding_row)}
product_image_embeddings = np.load(PRODUCT_EMBEDDING_DIR / "image_embeddings.npy")
query_image_embeddings = np.load(QUERY_EMBEDDING_DIR / "embeddings.npy")

print("queries:", len(queries))
print("products:", len(products))
print("metrics:", report["metrics"])
print("product image embeddings:", product_image_embeddings.shape)
print("query image embeddings:", query_image_embeddings.shape)

In [ ]:
def html_escape(value: object) -> str:
    return (
        str(value or "")
        .replace("&", "&amp;")
        .replace("<", "&lt;")
        .replace(">", "&gt;")
        .replace('"', "&quot;")
    )


def product_summary(product_id: str) -> dict:
    record = products[product_id]
    return {
        "product_id": product_id,
        "name": record.get("name"),
        "brand": record.get("brand_or_distillery"),
        "style": record.get("style"),
        "region": record.get("region"),
        "country": record.get("country"),
        "abv": record.get("abv"),
        "description": record.get("description"),
        "image_path": record.get("cropped_product_image_path") or record.get("product_image_path"),
    }


def first_relevant_rank(result: dict) -> int | None:
    relevant = set(result["relevant_product_ids"])
    for index, product_id in enumerate(result["retrieved_product_ids"], start=1):
        if product_id in relevant:
            return index
    return None


def similarity_score(query_id: str, product_id: str) -> float:
    query_vector = query_image_embeddings[query_embedding_rows[query_id]]
    product_vector = product_image_embeddings[product_embedding_rows[product_id]]
    denominator = np.linalg.norm(query_vector) * np.linalg.norm(product_vector)
    if denominator == 0:
        return 0.0
    return float(query_vector @ product_vector / denominator)


def display_local_image(path_value: str | None, width: int = 220) -> None:
    if not path_value:
        return
    path = Path(path_value)
    if not path.is_absolute():
        path = ROOT / path
    if path.exists():
        display(Image(filename=str(path), width=width))
    else:
        display(HTML(f"<small>Missing image: {html_escape(path_value)}</small>"))

## Query Browser

Change `query_id` below to inspect one image query. Query IDs ending in `_crop` use the manually cropped bottle image; query IDs ending in `_scene` use the full movie/TV scene image.

In [ ]:
query_id = "iq008_crop"
top_k = 5

In [ ]:
def show_query_results(query_id: str, top_k: int = 5, show_images: bool = True) -> None:
    result = next(row for row in report["query_results"] if row["query_id"] == query_id)
    query = queries[query_id]
    relevant = set(result["relevant_product_ids"])
    first_rank = first_relevant_rank(result)

    display(
        HTML(
            f"""
            <h2>{html_escape(query_id)} · {html_escape(query['query_style'])}</h2>
            <p><b>Screen:</b> {html_escape(query.get('screen_reference'))} &nbsp;
            <b>Whisky reference:</b> {html_escape(query.get('whisky_reference'))}</p>
            <p><b>First relevant rank:</b> {html_escape(first_rank)} &nbsp;
            <b>Relevant labels:</b> {len(relevant)}</p>
            <p><b>Notes:</b> {html_escape(query.get('notes'))}</p>
            <p><b>Query image:</b></p>
            """
        )
    )
    display_local_image(query.get("query_image_path"), width=90)

    rows = []
    for rank, product_id in enumerate(result["retrieved_product_ids"][:top_k], start=1):
        product = product_summary(product_id)
        marker = "✅ relevant" if product_id in relevant else "—"
        score = similarity_score(query_id, product_id)
        rows.append(
            f"""
            <tr>
              <td>{rank}</td>
              <td>{marker}</td>
              <td>{score:.4f}</td>
              <td><code>{html_escape(product_id)}</code></td>
              <td><b>{html_escape(product['name'])}</b><br>
              {html_escape(product['brand'])} · {html_escape(product['style'])} ·
              {html_escape(product['region'])} · {html_escape(product['country'])} ·
              {html_escape(product['abv'])}% ABV<br>
              <small>{html_escape(product['description'])}</small></td>
            </tr>
            """
        )

    display(
        HTML(
            """
            <table>
              <thead><tr><th>Rank</th><th>Hit</th><th>Score</th><th>Product ID</th><th>Product</th></tr></thead>
              <tbody>
            """
            + "\n".join(rows)
            + "</tbody></table>"
        )
    )

    if show_images:
        for rank, product_id in enumerate(result["retrieved_product_ids"][:top_k], start=1):
            product = product_summary(product_id)
            display(HTML(f"<b>Rank {rank}: {html_escape(product['name'])}</b>"))
            display_local_image(product.get("image_path"), width=90)


In [ ]:
# for cropped only
for query_id in queries.keys():
    if "crop" in query_id:
        show_query_results(query_id, top_k=5, show_images=True)

In [ ]:
# for whole only
for query_id in queries.keys():
    if "crop" not in query_id:
        show_query_results(query_id, top_k=5, show_images=True)

## All Image Query Summary

In [ ]:
summary_rows = []
for result in report["query_results"]:
    query = queries[result["query_id"]]
    first_rank = first_relevant_rank(result)
    top_products = result["retrieved_product_ids"][:5]
    top_names = [products[pid].get("name") for pid in top_products]
    top_scores = [f"{similarity_score(result['query_id'], pid):.4f}" for pid in top_products]
    summary_rows.append(
        f"""
        <tr>
          <td><code>{html_escape(result['query_id'])}</code></td>
          <td>{html_escape(query['query_style'])}</td>
          <td>{html_escape(query.get('screen_reference'))}</td>
          <td>{html_escape(query.get('whisky_reference'))}</td>
          <td>{html_escape(first_rank)}</td>
          <td>{html_escape(' | '.join(top_names))}</td>
          <td>{html_escape(' | '.join(top_scores))}</td>
        </tr>
        """
    )

display(
    HTML(
        """
        <table>
          <thead>
            <tr><th>Query ID</th><th>Style</th><th>Screen</th><th>Reference</th><th>First Relevant Rank</th><th>Top 5 Names</th><th>Top 5 Scores</th></tr>
          </thead>
          <tbody>
        """
        + "\n".join(summary_rows)
        + "</tbody></table>"
    )
)